# 01 — Feasibility and source-data audit

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


# AKI V2 — eICU Fizibilite Çalıştırıcısı

Bu notebook model eğitmez. eICU-CRD üzerinde toplulaştırılmış fizibilite denetimleri çalıştırır ve sonuçları Google Drive'a kaydeder.

**Yalnızca `PROJECT_ID` değerini değiştirin, sonra Runtime > Run all seçin.**


In [ ]:
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = "physionet-data.eicu_crd"
BQ_LOCATION = "US"
DRIVE_OUTPUT_DIR = f"{OUTPUT_ROOT}/01_FEASIBILITY_OUTPUTS"


In [ ]:
!pip -q install google-cloud-bigquery pandas pyarrow openpyxl
from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd
import numpy as np
import os, json, zipfile, hashlib, datetime, pathlib

auth.authenticate_user()
drive.mount('/content/drive')
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("BigQuery client ready:", PROJECT_ID)


## 01_table_inventory.sql


In [ ]:
QUERY_01_TABLE_INVENTORY = r"""-- Amaç: Gerekli eICU tablolarına erişimi ve satır sayılarını doğrulamak.
SELECT
  table_id AS table_name,
  row_count,
  size_bytes
FROM `{{SOURCE_DATASET}}.__TABLES__`
WHERE table_id IN (
  'patient','lab','intakeOutput','treatment','hospital',
  'vitalPeriodic','vitalAperiodic','nurseCharting',
  'apacheApsVar','apachePatientResult','apachePredVar'
)
ORDER BY table_id;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_01_table_inventory = client.query(QUERY_01_TABLE_INVENTORY).to_dataframe()
display(result_01_table_inventory.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "01_table_inventory.csv")
result_01_table_inventory.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_01_table_inventory))


## 02_patient_stay_audit.sql


In [ ]:
QUERY_02_PATIENT_STAY_AUDIT = r"""-- Amaç: Hasta, hastane, yatış ve takip kapsamını toplulaştırılmış olarak değerlendirmek.
WITH p AS (
  SELECT
    patientUnitStayID,
    patientHealthSystemStayID,
    uniquepid,
    hospitalID,
    unitVisitNumber,
    unitStayType,
    hospitalAdmitOffset,
    hospitalDischargeOffset,
    unitDischargeOffset,
    hospitalDischargeStatus,
    unitDischargeStatus,
    CASE
      WHEN age = '> 89' THEN 90
      ELSE SAFE_CAST(age AS INT64)
    END AS age_num
  FROM `{{SOURCE_DATASET}}.patient`
)
SELECT
  COUNT(*) AS total_icu_stays,
  COUNT(DISTINCT uniquepid) AS unique_patients,
  COUNT(DISTINCT patientHealthSystemStayID) AS hospital_stays,
  COUNT(DISTINCT hospitalID) AS hospitals,
  COUNTIF(age_num >= 18) AS adult_icu_stays,
  COUNTIF(age_num IS NULL) AS age_unresolved,
  COUNTIF(age_num >= 18 AND unitDischargeOffset >= 720) AS adult_stays_at_least_12h,
  COUNTIF(age_num >= 18 AND unitDischargeOffset >= 4320) AS adult_stays_at_least_72h,
  COUNTIF(age_num >= 18 AND unitDischargeOffset < 4320 AND unitDischargeStatus = 'Alive') AS live_icu_discharge_before_72h,
  COUNTIF(age_num >= 18 AND unitDischargeOffset < 4320 AND unitDischargeStatus = 'Expired') AS icu_death_before_72h,
  COUNTIF(age_num >= 18 AND unitVisitNumber = 1) AS adult_first_unit_visits,
  COUNTIF(age_num >= 18 AND unitVisitNumber > 1) AS adult_repeat_unit_visits
FROM p;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_02_patient_stay_audit = client.query(QUERY_02_PATIENT_STAY_AUDIT).to_dataframe()
display(result_02_patient_stay_audit.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "02_patient_stay_audit.csv")
result_02_patient_stay_audit.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_02_patient_stay_audit))


## 03_creatinine_name_inventory.sql


In [ ]:
QUERY_03_CREATININE_NAME_INVENTORY = r"""-- Amaç: Kreatinin için kullanılan labName ve birim varyantlarını belirlemek.
SELECT
  LOWER(TRIM(labName)) AS lab_name,
  labMeasureNameSystem AS unit_system,
  labMeasureNameInterface AS unit_interface,
  COUNT(*) AS measurement_count,
  COUNT(DISTINCT patientUnitStayID) AS patient_stays
FROM `{{SOURCE_DATASET}}.lab`
WHERE REGEXP_CONTAINS(LOWER(labName), r'creat')
GROUP BY 1,2,3
ORDER BY measurement_count DESC;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_03_creatinine_name_inventory = client.query(QUERY_03_CREATININE_NAME_INVENTORY).to_dataframe()
display(result_03_creatinine_name_inventory.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "03_creatinine_name_inventory.csv")
result_03_creatinine_name_inventory.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_03_creatinine_name_inventory))


## 04_creatinine_window_coverage.sql


In [ ]:
QUERY_04_CREATININE_WINDOW_COVERAGE = r"""-- Amaç: Aday referans ve outcome pencerelerinde kreatinin ölçüm kapsamını değerlendirmek.
WITH adults AS (
  SELECT
    patientUnitStayID,
    hospitalAdmitOffset,
    unitDischargeOffset,
    CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END AS age_num
  FROM `{{SOURCE_DATASET}}.patient`
),
creat AS (
  SELECT
    patientUnitStayID,
    labResultOffset,
    SAFE_CAST(labResult AS FLOAT64) AS creatinine
  FROM `{{SOURCE_DATASET}}.lab`
  WHERE LOWER(TRIM(labName)) = 'creatinine'
    AND SAFE_CAST(labResult AS FLOAT64) BETWEEN 0.1 AND 30.0
),
flags AS (
  SELECT
    a.patientUnitStayID,
    a.unitDischargeOffset,
    COUNTIF(c.labResultOffset BETWEEN GREATEST(a.hospitalAdmitOffset, -1440) AND 360) AS n_ref_m24_to_p6,
    COUNTIF(c.labResultOffset BETWEEN a.hospitalAdmitOffset AND 360) AS n_ref_hosp_to_p6,
    COUNTIF(c.labResultOffset BETWEEN 0 AND 720) AS n_creat_0_12h,
    COUNTIF(c.labResultOffset > 720 AND c.labResultOffset <= 2880) AS n_creat_12_48h,
    COUNTIF(c.labResultOffset > 720 AND c.labResultOffset <= 4320) AS n_creat_12_72h
  FROM adults a
  LEFT JOIN creat c USING (patientUnitStayID)
  WHERE a.age_num >= 18 AND a.unitDischargeOffset >= 720
  GROUP BY 1,2
)
SELECT
  COUNT(*) AS adult_stays_at_prediction_time,
  COUNTIF(n_ref_m24_to_p6 > 0) AS with_reference_m24_to_p6,
  COUNTIF(n_ref_hosp_to_p6 > 0) AS with_reference_hosp_to_p6,
  COUNTIF(n_creat_0_12h > 0) AS with_creatinine_0_12h,
  COUNTIF(n_creat_12_48h > 0) AS with_creatinine_12_48h,
  COUNTIF(n_creat_12_72h > 0) AS with_creatinine_12_72h,
  COUNTIF(n_creat_12_72h = 0 AND unitDischargeOffset >= 4320) AS no_future_creat_despite_72h_icu,
  APPROX_QUANTILES(n_creat_0_12h, 10) AS deciles_n_creat_0_12h,
  APPROX_QUANTILES(n_creat_12_72h, 10) AS deciles_n_creat_12_72h
FROM flags;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_04_creatinine_window_coverage = client.query(QUERY_04_CREATININE_WINDOW_COVERAGE).to_dataframe()
display(result_04_creatinine_window_coverage.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "04_creatinine_window_coverage.csv")
result_04_creatinine_window_coverage.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_04_creatinine_window_coverage))


## 05_candidate_kdigo_event_counts.sql


In [ ]:
QUERY_05_CANDIDATE_KDIGO_EVENT_COUNTS = r"""-- Amaç: İki referans kreatinin tanımıyla aday KDIGO evre 2–3 olay sayılarını karşılaştırmak.
WITH adults AS (
  SELECT
    patientUnitStayID,
    hospitalID,
    hospitalAdmitOffset,
    unitDischargeOffset,
    CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END AS age_num
  FROM `{{SOURCE_DATASET}}.patient`
  WHERE (CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END) >= 18
    AND unitDischargeOffset >= 720
),
creat AS (
  SELECT
    patientUnitStayID,
    labResultOffset,
    SAFE_CAST(labResult AS FLOAT64) AS creatinine
  FROM `{{SOURCE_DATASET}}.lab`
  WHERE LOWER(TRIM(labName)) = 'creatinine'
    AND SAFE_CAST(labResult AS FLOAT64) BETWEEN 0.1 AND 30.0
),
summary AS (
  SELECT
    a.patientUnitStayID,
    a.hospitalID,
    a.unitDischargeOffset,
    ARRAY_AGG(IF(c.labResultOffset BETWEEN GREATEST(a.hospitalAdmitOffset,-1440) AND 360,
                 STRUCT(c.labResultOffset AS off, c.creatinine AS val), NULL)
              IGNORE NULLS ORDER BY c.labResultOffset LIMIT 1)[SAFE_OFFSET(0)].val AS baseline_first_m24_p6,
    MIN(IF(c.labResultOffset BETWEEN a.hospitalAdmitOffset AND 720, c.creatinine, NULL)) AS baseline_min_hosp_p12,
    MAX(IF(c.labResultOffset BETWEEN 0 AND 720, c.creatinine, NULL)) AS max_creat_0_12h,
    MAX(IF(c.labResultOffset > 720 AND c.labResultOffset <= 4320, c.creatinine, NULL)) AS max_creat_12_72h,
    COUNTIF(c.labResultOffset > 720 AND c.labResultOffset <= 4320) AS n_future_creat
  FROM adults a
  LEFT JOIN creat c USING (patientUnitStayID)
  GROUP BY 1,2,3
),
classified AS (
  SELECT *,
    CASE WHEN baseline_first_m24_p6 IS NULL OR max_creat_0_12h IS NULL THEN NULL
         WHEN max_creat_0_12h >= 2.0 * baseline_first_m24_p6
           OR (max_creat_0_12h >= 4.0 AND max_creat_0_12h - baseline_first_m24_p6 >= 0.5)
         THEN 1 ELSE 0 END AS existing_stage23_first,
    CASE WHEN baseline_min_hosp_p12 IS NULL OR max_creat_0_12h IS NULL THEN NULL
         WHEN max_creat_0_12h >= 2.0 * baseline_min_hosp_p12
           OR (max_creat_0_12h >= 4.0 AND max_creat_0_12h - baseline_min_hosp_p12 >= 0.5)
         THEN 1 ELSE 0 END AS existing_stage23_min,
    CASE WHEN baseline_first_m24_p6 IS NULL OR max_creat_12_72h IS NULL THEN NULL
         WHEN max_creat_12_72h >= 2.0 * baseline_first_m24_p6
           OR (max_creat_12_72h >= 4.0 AND max_creat_12_72h - baseline_first_m24_p6 >= 0.5)
         THEN 1 ELSE 0 END AS future_stage23_first,
    CASE WHEN baseline_min_hosp_p12 IS NULL OR max_creat_12_72h IS NULL THEN NULL
         WHEN max_creat_12_72h >= 2.0 * baseline_min_hosp_p12
           OR (max_creat_12_72h >= 4.0 AND max_creat_12_72h - baseline_min_hosp_p12 >= 0.5)
         THEN 1 ELSE 0 END AS future_stage23_min
  FROM summary
)
SELECT
  COUNT(*) AS adult_stays_at_12h,
  COUNTIF(baseline_first_m24_p6 IS NOT NULL) AS baseline_first_available,
  COUNTIF(baseline_min_hosp_p12 IS NOT NULL) AS baseline_min_available,
  COUNTIF(n_future_creat > 0) AS future_creat_available,
  COUNTIF(existing_stage23_first = 1) AS existing_stage23_first_definition,
  COUNTIF(existing_stage23_min = 1) AS existing_stage23_min_definition,
  COUNTIF(existing_stage23_first = 0 AND future_stage23_first = 1) AS incident_stage23_first_definition,
  COUNTIF(existing_stage23_min = 0 AND future_stage23_min = 1) AS incident_stage23_min_definition,
  SAFE_DIVIDE(COUNTIF(existing_stage23_first = 0 AND future_stage23_first = 1),
              COUNTIF(existing_stage23_first = 0 AND future_stage23_first IS NOT NULL)) AS incidence_first_definition,
  SAFE_DIVIDE(COUNTIF(existing_stage23_min = 0 AND future_stage23_min = 1),
              COUNTIF(existing_stage23_min = 0 AND future_stage23_min IS NOT NULL)) AS incidence_min_definition
FROM classified;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_05_candidate_kdigo_event_counts = client.query(QUERY_05_CANDIDATE_KDIGO_EVENT_COUNTS).to_dataframe()
display(result_05_candidate_kdigo_event_counts.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "05_candidate_kdigo_event_counts.csv")
result_05_candidate_kdigo_event_counts.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_05_candidate_kdigo_event_counts))


## 06_followup_and_discharge_audit.sql


In [ ]:
QUERY_06_FOLLOWUP_AND_DISCHARGE_AUDIT = r"""-- Amaç: Outcome gözlenebilirliği, erken taburculuk ve erken ölüm gruplarını belirlemek.
WITH adults AS (
  SELECT
    patientUnitStayID,
    unitDischargeOffset,
    unitDischargeStatus,
    hospitalDischargeOffset,
    hospitalDischargeStatus,
    CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END AS age_num
  FROM `{{SOURCE_DATASET}}.patient`
),
future_labs AS (
  SELECT
    patientUnitStayID,
    COUNTIF(labResultOffset > 720 AND labResultOffset <= 4320
            AND LOWER(TRIM(labName)) = 'creatinine'
            AND SAFE_CAST(labResult AS FLOAT64) BETWEEN 0.1 AND 30.0) AS n_future_creat,
    MAX(IF(labResultOffset > 720 AND labResultOffset <= 4320
           AND LOWER(TRIM(labName)) = 'creatinine', labResultOffset, NULL)) AS last_future_creat_offset
  FROM `{{SOURCE_DATASET}}.lab`
  GROUP BY patientUnitStayID
)
SELECT
  COUNT(*) AS adult_stays_reaching_12h,
  COUNTIF(a.unitDischargeOffset >= 4320) AS remained_in_icu_72h,
  COUNTIF(a.unitDischargeOffset < 4320 AND a.unitDischargeStatus = 'Alive') AS live_icu_discharge_before_72h,
  COUNTIF(a.unitDischargeOffset < 4320 AND a.unitDischargeStatus = 'Expired') AS icu_death_before_72h,
  COUNTIF(COALESCE(f.n_future_creat,0) > 0) AS any_future_creatinine,
  COUNTIF(COALESCE(f.n_future_creat,0) = 0) AS no_future_creatinine,
  COUNTIF(COALESCE(f.n_future_creat,0) = 0 AND a.unitDischargeOffset >= 4320) AS no_future_creat_but_icu_72h,
  COUNTIF(COALESCE(f.n_future_creat,0) = 0 AND a.unitDischargeOffset < 4320 AND a.unitDischargeStatus = 'Alive') AS no_future_creat_live_discharge,
  COUNTIF(COALESCE(f.n_future_creat,0) = 0 AND a.unitDischargeOffset < 4320 AND a.unitDischargeStatus = 'Expired') AS no_future_creat_early_death
FROM adults a
LEFT JOIN future_labs f USING (patientUnitStayID)
WHERE a.age_num >= 18 AND a.unitDischargeOffset >= 720;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_06_followup_and_discharge_audit = client.query(QUERY_06_FOLLOWUP_AND_DISCHARGE_AUDIT).to_dataframe()
display(result_06_followup_and_discharge_audit.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "06_followup_and_discharge_audit.csv")
result_06_followup_and_discharge_audit.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_06_followup_and_discharge_audit))


## 07_rrt_dialysis_inventory.sql


In [ ]:
QUERY_07_RRT_DIALYSIS_INVENTORY = r"""-- Amaç: RRT/diyaliz için pozitif kayıt terimlerini ve kapsamı keşfetmek; outcome henüz üretilmez.
WITH treatment_terms AS (
  SELECT
    LOWER(TRIM(treatmentString)) AS term,
    COUNT(*) AS records,
    COUNT(DISTINCT patientUnitStayID) AS patient_stays
  FROM `{{SOURCE_DATASET}}.treatment`
  WHERE REGEXP_CONTAINS(LOWER(treatmentString), r'dialy|renal replacement|hemofil|cvvh|cvvhd|cvvhdf|crrt')
  GROUP BY 1
),
dialysis_io AS (
  SELECT
    COUNT(*) AS io_records_with_dialysis_total,
    COUNT(DISTINCT patientUnitStayID) AS io_patient_stays_with_dialysis_total,
    COUNTIF(dialysisTotal != 0) AS io_nonzero_dialysis_records,
    COUNT(DISTINCT IF(dialysisTotal != 0, patientUnitStayID, NULL)) AS io_patient_stays_nonzero_dialysis
  FROM `{{SOURCE_DATASET}}.intakeOutput`
  WHERE dialysisTotal IS NOT NULL
)
SELECT 'TREATMENT_TERM' AS source, term AS item,
       records AS record_count, patient_stays AS patient_stay_count
FROM treatment_terms
UNION ALL
SELECT 'INTAKEOUTPUT_SUMMARY', 'dialysisTotal not null',
       io_records_with_dialysis_total, io_patient_stays_with_dialysis_total
FROM dialysis_io
UNION ALL
SELECT 'INTAKEOUTPUT_SUMMARY', 'dialysisTotal nonzero',
       io_nonzero_dialysis_records, io_patient_stays_nonzero_dialysis
FROM dialysis_io
ORDER BY source, record_count DESC;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_07_rrt_dialysis_inventory = client.query(QUERY_07_RRT_DIALYSIS_INVENTORY).to_dataframe()
display(result_07_rrt_dialysis_inventory.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "07_rrt_dialysis_inventory.csv")
result_07_rrt_dialysis_inventory.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_07_rrt_dialysis_inventory))


## 08_urine_output_inventory.sql


In [ ]:
QUERY_08_URINE_OUTPUT_INVENTORY = r"""-- Amaç: İdrar çıkışı kayıt terimlerini ve hasta kapsamını keşfetmek.
SELECT
  LOWER(TRIM(cellLabel)) AS cell_label,
  LOWER(TRIM(cellPath)) AS cell_path,
  COUNT(*) AS records,
  COUNT(DISTINCT patientUnitStayID) AS patient_stays,
  APPROX_QUANTILES(SAFE_CAST(cellValueNumeric AS FLOAT64), 10) AS value_deciles
FROM `{{SOURCE_DATASET}}.intakeOutput`
WHERE REGEXP_CONTAINS(LOWER(CONCAT(cellLabel, ' ', cellPath)), r'urine|urinary|foley|void|urostomy|nephrostomy')
GROUP BY 1,2
ORDER BY records DESC
LIMIT 500;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_08_urine_output_inventory = client.query(QUERY_08_URINE_OUTPUT_INVENTORY).to_dataframe()
display(result_08_urine_output_inventory.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "08_urine_output_inventory.csv")
result_08_urine_output_inventory.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_08_urine_output_inventory))


## 09_hospital_feasibility.sql


In [ ]:
QUERY_09_HOSPITAL_FEASIBILITY = r"""-- Amaç: Hastane bazında örneklem ve aday olay sayısını değerlendirmek.
-- Bu çıktı hospitalID içerir; yetkili ortamda tutulmalıdır.
WITH adults AS (
  SELECT
    patientUnitStayID,
    hospitalID,
    hospitalAdmitOffset,
    unitDischargeOffset,
    CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END AS age_num
  FROM `{{SOURCE_DATASET}}.patient`
  WHERE (CASE WHEN age = '> 89' THEN 90 ELSE SAFE_CAST(age AS INT64) END) >= 18
    AND unitDischargeOffset >= 720
),
creat AS (
  SELECT
    patientUnitStayID,
    labResultOffset,
    SAFE_CAST(labResult AS FLOAT64) AS creatinine
  FROM `{{SOURCE_DATASET}}.lab`
  WHERE LOWER(TRIM(labName)) = 'creatinine'
    AND SAFE_CAST(labResult AS FLOAT64) BETWEEN 0.1 AND 30.0
),
s AS (
  SELECT
    a.patientUnitStayID,
    a.hospitalID,
    ARRAY_AGG(IF(c.labResultOffset BETWEEN GREATEST(a.hospitalAdmitOffset,-1440) AND 360,
                 STRUCT(c.labResultOffset AS off, c.creatinine AS val), NULL)
              IGNORE NULLS ORDER BY c.labResultOffset LIMIT 1)[SAFE_OFFSET(0)].val AS baseline,
    MAX(IF(c.labResultOffset BETWEEN 0 AND 720, c.creatinine, NULL)) AS max_0_12,
    MAX(IF(c.labResultOffset > 720 AND c.labResultOffset <= 4320, c.creatinine, NULL)) AS max_12_72
  FROM adults a
  LEFT JOIN creat c USING(patientUnitStayID)
  GROUP BY 1,2
),
c AS (
  SELECT *,
    CASE WHEN baseline IS NULL OR max_0_12 IS NULL THEN NULL
         WHEN max_0_12 >= 2*baseline OR (max_0_12 >= 4 AND max_0_12-baseline >= 0.5) THEN 1 ELSE 0 END AS existing,
    CASE WHEN baseline IS NULL OR max_12_72 IS NULL THEN NULL
         WHEN max_12_72 >= 2*baseline OR (max_12_72 >= 4 AND max_12_72-baseline >= 0.5) THEN 1 ELSE 0 END AS future_event
  FROM s
)
SELECT
  hospitalID,
  COUNT(*) AS adult_stays_reaching_12h,
  COUNTIF(baseline IS NOT NULL) AS baseline_available,
  COUNTIF(max_12_72 IS NOT NULL) AS future_creat_available,
  COUNTIF(existing = 0 AND future_event IS NOT NULL) AS analyzable_incident_cohort,
  COUNTIF(existing = 0 AND future_event = 1) AS candidate_events,
  SAFE_DIVIDE(COUNTIF(existing = 0 AND future_event = 1),
              COUNTIF(existing = 0 AND future_event IS NOT NULL)) AS candidate_event_rate
FROM c
GROUP BY hospitalID
ORDER BY analyzable_incident_cohort DESC;
""".replace("{{SOURCE_DATASET}}", SOURCE_DATASET)
result_09_hospital_feasibility = client.query(QUERY_09_HOSPITAL_FEASIBILITY).to_dataframe()
display(result_09_hospital_feasibility.head(50))
out_path = os.path.join(DRIVE_OUTPUT_DIR, "09_hospital_feasibility.csv")
result_09_hospital_feasibility.to_csv(out_path, index=False)
print("Saved:", out_path, "rows=", len(result_09_hospital_feasibility))


## Kimliksiz hastane özeti ve paketleme

Aşağıdaki hücre hospitalID değerlerini dışa aktarmadan hastane büyüklüğü ve olay sayısı dağılımlarını özetler.


In [ ]:
hosp = result_09_hospital_feasibility.copy()
summary = pd.DataFrame({
    "metric": [
        "hospital_count", "median_analyzable_cohort", "q25_analyzable_cohort",
        "q75_analyzable_cohort", "hospitals_with_at_least_20_events",
        "hospitals_with_at_least_50_events", "total_candidate_events"
    ],
    "value": [
        len(hosp), hosp['analyzable_incident_cohort'].median(),
        hosp['analyzable_incident_cohort'].quantile(0.25),
        hosp['analyzable_incident_cohort'].quantile(0.75),
        int((hosp['candidate_events'] >= 20).sum()),
        int((hosp['candidate_events'] >= 50).sum()),
        int(hosp['candidate_events'].sum())
    ]
})
summary_path = os.path.join(DRIVE_OUTPUT_DIR, "09_hospital_feasibility_DEIDENTIFIED_SUMMARY.csv")
summary.to_csv(summary_path, index=False)
display(summary)

# Manifest ve ZIP: hospitalID içeren ayrıntılı dosya ZIP'e alınmaz.
manifest=[]
for path in pathlib.Path(DRIVE_OUTPUT_DIR).glob('*.csv'):
    if path.name == '09_hospital_feasibility.csv':
        continue
    data=path.read_bytes()
    manifest.append({"filename":path.name,"size_bytes":len(data),"sha256":hashlib.sha256(data).hexdigest()})
manifest_path=pathlib.Path(DRIVE_OUTPUT_DIR)/"MANIFEST.json"
manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
zip_path=pathlib.Path(DRIVE_OUTPUT_DIR)/"AKI_V2_FEASIBILITY_OUTPUTS.zip"
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for item in manifest:
        z.write(pathlib.Path(DRIVE_OUTPUT_DIR)/item['filename'], arcname=item['filename'])
    z.write(manifest_path, arcname='MANIFEST.json')
print("Shareable aggregate package:", zip_path)
print("Do NOT upload 09_hospital_feasibility.csv; it remains in your authorized Drive environment.")
